# Visium HD isocortex, SMS versus CTR

Same analysis as the files in `R/`. Set the working directory to this folder before running the cells from top to bottom. Figures and tables go to `results/`.

Counts and the GSE124952 reference are not in the repository. Notes on seeds, the two DiSTect coordinate systems, and the two sections that have no script are in `README.md`.


## 1. Paths, boxes, and depth cuts

`R/00_config.R`


In [ ]:
# Edit proj_dir, then source the numbered scripts from this directory.
#   source("R/01_import_isocortex.R")

# Analysis-machine project directory. The escapes keep the path valid
# regardless of how R reads this file.
proj_dir <- "F:/H.M.\u6253\u706b\u673a"

repo_dir <- normalizePath(getwd(), winslash = "/")
results_dir <- file.path(repo_dir, "results")

path_ctr <- file.path(proj_dir, "Huang_Minza__Ctr_1_A1", "square_016um")
path_sms <- file.path(proj_dir, "Huang_Minza__SMS_1_D1", "square_016um")
path_ref_counts <- file.path(proj_dir, "Task2", "DataFromPaper", "GSE124952_expression_matrix.csv.gz")
path_ref_meta <- file.path(proj_dir, "Task2", "DataFromPaper", "GSE124952_meta_data.csv.gz")

ctr_boxes <- data.frame(
  xmin = c(4000, 15500), xmax = c(14500, 27500),
  ymin = c(7000, 6000), ymax = c(17500, 16500)
)
sms_boxes <- data.frame(
  xmin = c(15000, 2000), xmax = c(26000, 14500),
  ymin = c(10000, 12500), ymax = c(19500, 21500)
)

# Drawn on the H&E before differential expression. In GetTissueCoordinates()
# for these captures, x is full-resolution row and y is full-resolution column.
expected_bins <- c(CTR_Rep1 = 27586L, CTR_Rep2 = 28126L, SMS_Rep1 = 26691L, SMS_Rep2 = 25575L)

cut_L1 <- 0.12
cut_L23 <- 0.42
cut_L5 <- 0.72
upper_max <- 0.42
deep_min <- 0.55

dir.create(results_dir, showWarnings = FALSE, recursive = TRUE)
options(warn = 1)
options(future.globals.maxSize = 8000 * 1024^2)

out_dir <- function(...) {
  d <- file.path(results_dir, ...)
  dir.create(d, showWarnings = FALSE, recursive = TRUE)
  d
}

load_cortex <- function() {
  p <- file.path(results_dir, "cortex.rds")
  if (!file.exists(p)) stop("cortex.rds is missing. Run R/01_import_isocortex.R from the repository root.")
  readRDS(p)
}

save_cortex <- function(obj) {
  saveRDS(obj, file.path(results_dir, "cortex.rds"))
  invisible(obj)
}


## 2. Import and crop

`R/01_import_isocortex.R`


In [ ]:
# Four isocortical pieces. Counts are joined, then log-normalized.
source("R/00_config.R")

suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(arrow))

crop_piece <- function(obj, boxes, sample_name, positions) {
  coords <- GetTissueCoordinates(obj)
  if ("cell" %in% colnames(coords)) rownames(coords) <- coords$cell
  x_col <- grep("col|x", colnames(coords), value = TRUE, ignore.case = TRUE)[1]
  y_col <- grep("row|y", colnames(coords), value = TRUE, ignore.case = TRUE)[1]
  if (is.na(x_col) || is.na(y_col) || x_col == y_col) {
    stop("Could not tell the coordinate columns apart.")
  }

  replicate <- rep(NA_character_, ncol(obj))
  names(replicate) <- colnames(obj)
  keep <- character()
  for (i in seq_len(nrow(boxes))) {
    ids <- rownames(coords[
      coords[[x_col]] >= boxes$xmin[i] & coords[[x_col]] <= boxes$xmax[i] &
        coords[[y_col]] >= boxes$ymin[i] & coords[[y_col]] <= boxes$ymax[i],
    ])
    if (length(ids)) {
      replicate[ids] <- paste0(sample_name, "_Rep", i)
      keep <- c(keep, ids)
    }
  }
  keep <- unique(keep)
  if (!length(keep)) stop("No bins in the ", sample_name, " boxes.")

  obj <- subset(obj, cells = keep)
  obj$spatial_replicate <- replicate[colnames(obj)]
  obj$condition <- sample_name
  obj$pxl_x <- coords[colnames(obj), x_col]
  obj$pxl_y <- coords[colnames(obj), y_col]

  hit <- match(colnames(obj), positions$barcode)
  if (anyNA(hit)) stop("Array coordinates missing for ", sum(is.na(hit)), " ", sample_name, " bins.")
  obj$array_row <- positions$array_row[hit]
  obj$array_col <- positions$array_col[hit]
  obj
}

read_positions <- function(sample_dir) {
  arrow::read_parquet(file.path(sample_dir, "spatial", "tissue_positions.parquet"))
}

ctr <- Load10X_Spatial(path_ctr, filename = "filtered_feature_bc_matrix.h5")
sms <- Load10X_Spatial(path_sms, filename = "filtered_feature_bc_matrix.h5")
ctr <- crop_piece(ctr, ctr_boxes, "CTR", read_positions(path_ctr))
sms <- crop_piece(sms, sms_boxes, "SMS", read_positions(path_sms))

cortex <- merge(ctr, y = sms, add.cell.ids = c("CTR", "SMS"))
rm(ctr, sms)
cortex <- JoinLayers(cortex)
DefaultAssay(cortex) <- "Spatial"
cortex <- NormalizeData(cortex, normalization.method = "LogNormalize", verbose = FALSE)
Idents(cortex) <- "condition"

counts <- table(cortex$spatial_replicate)
print(counts)
if (!identical(as.integer(counts[names(expected_bins)]), as.integer(expected_bins))) {
  stop("Crop bin counts do not match the analysis set.")
}
message("Bins: ", ncol(cortex), "  Genes: ", nrow(cortex))
save_cortex(cortex)


## 3. Pseudobulk DESeq2

`R/02_pseudobulk.R`


In [ ]:
# One pseudobulk profile per animal. CTR is the reference, so a positive
# log2 fold change is higher in SMS. Drd2 and Adora2a stay in the model
# and are removed only from the saved table.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(DESeq2)
  library(apeglm)
  library(ggplot2)
  library(ggrepel)
  library(clusterProfiler)
  library(org.Mm.eg.db)
})

cortex <- load_cortex()
out <- out_dir("pseudobulk")

medians <- data.frame(
  replicate = names(tapply(cortex$nCount_Spatial, cortex$spatial_replicate, median)),
  median_umis = as.numeric(tapply(cortex$nCount_Spatial, cortex$spatial_replicate, median)),
  median_genes = as.numeric(tapply(cortex$nFeature_Spatial, cortex$spatial_replicate, median))
)
write.csv(medians, file.path(out, "per_replicate_medians.csv"), row.names = FALSE)

run_pseudobulk <- function(obj) {
  # AggregateExpression already sums the counts layer. Passing layer=
  # again is an error in Seurat 5.3.
  pseudo <- AggregateExpression(
    obj,
    group.by = "spatial_replicate",
    assays = "Spatial",
    return.seurat = FALSE
  )
  cts <- round(pseudo$Spatial)
  coldata <- data.frame(
    row.names = colnames(cts),
    condition = factor(
      ifelse(grepl("^CTR", colnames(cts)), "CTR", "SMS"),
      levels = c("CTR", "SMS")
    )
  )
  dds <- DESeqDataSetFromMatrix(cts, coldata, design = ~ condition)
  dds <- dds[rowSums(counts(dds) >= 10) >= 2, ]
  dds <- DESeq(dds)
  coef_name <- "condition_SMS_vs_CTR"
  if (!coef_name %in% resultsNames(dds)) {
    stop("DESeq2 coefficient ", coef_name, " was not found.")
  }
  res_base <- results(dds, contrast = c("condition", "SMS", "CTR"))
  shrunk <- lfcShrink(dds, coef = coef_name, type = "apeglm", res = res_base)
  df <- as.data.frame(shrunk)
  df$gene <- rownames(df)
  list(dds = dds, table = df)
}

full <- run_pseudobulk(cortex)
res <- full$table[!full$table$gene %in% c("Drd2", "Adora2a"), ]
write.csv(res, file.path(out, "pseudobulk_deseq2_shrunken.csv"), row.names = FALSE)

ieg <- c("Rai1", "Fos", "Junb", "Fosl2", "Egr1", "Egr2", "Nr4a1", "Nr4a2", "Nr4a3", "Arc", "Dusp5")
res$diffexpressed <- "NO"
res$diffexpressed[res$log2FoldChange > 0 & res$padj < 0.05] <- "UP"
res$diffexpressed[res$log2FoldChange < 0 & res$padj < 0.05] <- "DOWN"
res$label <- ifelse(res$gene %in% ieg, res$gene, NA)

volcano <- ggplot(res, aes(log2FoldChange, -log10(padj), col = diffexpressed, label = label)) +
  geom_point(alpha = 0.6, size = 2) +
  scale_color_manual(values = c(DOWN = "blue", NO = "grey", UP = "red")) +
  geom_vline(xintercept = 0, linetype = "dashed") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
  geom_text_repel(
    data = subset(res, !is.na(label)),
    max.overlaps = Inf, box.padding = 0.8, color = "black", size = 3
  ) +
  theme_minimal() +
  labs(
    x = "Shrunken log2 fold change",
    y = "-log10 adjusted p",
    color = NULL,
    title = "Isocortex pseudobulk, SMS versus CTR"
  ) +
  theme(legend.position = "none")
ggsave(file.path(out, "pseudobulk_volcano.png"), volcano, width = 8, height = 6, dpi = 140)

vsd <- vst(full$dds, blind = TRUE, nsub = min(1000, nrow(full$dds)))
pca <- plotPCA(vsd, intgroup = "condition") +
  geom_text_repel(aes(label = name), size = 3.5) +
  scale_color_manual(values = c(CTR = "grey40", SMS = "red")) +
  theme_minimal() +
  labs(title = "Pseudobulk PCA")
ggsave(file.path(out, "pseudobulk_pca.png"), pca, width = 6, height = 5, dpi = 140)

# Sensitivity: drop bins below 100 UMIs or 50 genes and repeat the same model.
pass <- cortex$nCount_Spatial >= 100 & cortex$nFeature_Spatial >= 50
removed <- !pass
qc <- data.frame(
  group = c("CTR", "SMS", "all"),
  bins_removed = c(
    sum(removed & cortex$condition == "CTR"),
    sum(removed & cortex$condition == "SMS"),
    sum(removed)
  ),
  bins_total = c(
    sum(cortex$condition == "CTR"),
    sum(cortex$condition == "SMS"),
    ncol(cortex)
  ),
  umis_removed = c(
    sum(cortex$nCount_Spatial[removed & cortex$condition == "CTR"]),
    sum(cortex$nCount_Spatial[removed & cortex$condition == "SMS"]),
    sum(cortex$nCount_Spatial[removed])
  ),
  umis_total = c(
    sum(cortex$nCount_Spatial[cortex$condition == "CTR"]),
    sum(cortex$nCount_Spatial[cortex$condition == "SMS"]),
    sum(cortex$nCount_Spatial)
  )
)
write.csv(qc, file.path(out, "qc_sensitivity_counts.csv"), row.names = FALSE)

filtered <- run_pseudobulk(subset(cortex, cells = colnames(cortex)[pass]))
filt <- filtered$table
filt <- filt[filt$gene %in% ieg, c("gene", "log2FoldChange", "lfcSE", "pvalue", "padj")]
write.csv(filt, file.path(out, "qc_sensitivity_ieg.csv"), row.names = FALSE)

# Whole-isocortex GO uses these pseudobulk calls (BH padj < 0.05), split by sign.
for (direction in c("UP", "DOWN")) {
  genes <- res$gene[res$diffexpressed == direction]
  if (length(genes) < 5) next
  ego <- enrichGO(
    genes, OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "BP",
    pAdjustMethod = "BH", pvalueCutoff = 0.05
  )
  if (is.null(ego) || nrow(ego) == 0) next
  write.csv(as.data.frame(ego), file.path(out, paste0("go_", tolower(direction), ".csv")), row.names = FALSE)
  p <- dotplot(ego, showCategory = 15) + ggtitle(paste("Isocortex", direction))
  ggsave(file.path(out, paste0("go_", tolower(direction), ".png")), p, width = 7, height = 6, dpi = 140)
}

p <- DotPlot(cortex, features = ieg, group.by = "condition") +
  ggtitle("Selected genes, all isocortical bins")
ggsave(file.path(out, "selected_genes_dotplot.png"), p, width = 8, height = 4, dpi = 140)
message("Pseudobulk tables are in ", out)


## 4. Label transfer

`R/03_label_transfer.R`


In [ ]:
# Label transfer from Bhattacherjee et al. 2019, GSE124952.
# The score is label confidence, not a cell count.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
})

if (!file.exists(path_ref_counts) || !file.exists(path_ref_meta)) {
  stop("GSE124952 counts or metadata not found under Task2/DataFromPaper.")
}

ref_counts <- read.csv(path_ref_counts, row.names = 1, check.names = FALSE)
ref_meta <- read.csv(path_ref_meta, row.names = 1)
ref <- CreateSeuratObject(counts = ref_counts, meta.data = ref_meta)
if (!"CellType" %in% colnames(ref[[]])) stop("Reference metadata has no CellType column.")
Idents(ref) <- ref$CellType
ref <- NormalizeData(ref, verbose = FALSE)
ref <- FindVariableFeatures(ref, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
ref <- ScaleData(ref, verbose = FALSE)
ref <- RunPCA(ref, verbose = FALSE)

cortex <- load_cortex()
DefaultAssay(cortex) <- "Spatial"
set.seed(1)
cortex <- FindVariableFeatures(cortex, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
cortex <- ScaleData(cortex, verbose = FALSE)
cortex <- RunPCA(cortex, verbose = FALSE)

anchors <- FindTransferAnchors(
  reference = ref, query = cortex,
  normalization.method = "LogNormalize", dims = 1:30, verbose = FALSE
)
pred <- TransferData(
  anchorset = anchors, refdata = ref$CellType,
  prediction.assay = TRUE, weight.reduction = cortex[["pca"]],
  dims = 1:30, verbose = FALSE
)
cortex[["prediction"]] <- pred
probs <- GetAssayData(cortex, assay = "prediction")
# TransferData appends a feature named max. It is not a class.
if ("max" %in% rownames(probs)) {
  probs <- probs[rownames(probs) != "max", , drop = FALSE]
}
cortex$predicted_celltype <- rownames(probs)[apply(probs, 2, which.max)]
cortex$prediction_score <- apply(probs, 2, max)
Idents(cortex) <- "predicted_celltype"

out <- out_dir("celltypes")
scores <- data.frame(
  bins = ncol(cortex),
  score_at_least_0.5 = sum(cortex$prediction_score >= 0.5)
)
write.csv(scores, file.path(out, "prediction_score_summary.csv"), row.names = FALSE)
write.csv(
  as.data.frame(table(cortex$predicted_celltype, cortex$spatial_replicate)),
  file.path(out, "celltype_counts_by_replicate.csv"),
  row.names = FALSE
)

counts <- as.data.frame(table(cortex$predicted_celltype, cortex$spatial_replicate))
names(counts) <- c("celltype", "replicate", "n")
counts$condition <- sub("_Rep[12]$", "", counts$replicate)
p <- ggplot(counts, aes(celltype, n, fill = condition, group = replicate)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7, color = "black", linewidth = 0.2) +
  scale_fill_manual(values = c(CTR = "grey80", SMS = "red")) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = NULL, y = "Bins", fill = NULL, title = "Predicted class by animal")
ggsave(file.path(out, "celltype_counts_by_replicate.png"), p, width = 9, height = 5, dpi = 140)

save_cortex(cortex)
message(sprintf(
  "Score >= 0.5: %d / %d",
  scores$score_at_least_0.5, scores$bins
))


## 5. Cell-class differential expression

`R/04_celltype_de.R`


In [ ]:
# Wilcoxon tests within each predicted class. p_val_adj is Seurat's Bonferroni.
# Drd2 and Adora2a are removed after the test.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(ggplot2)
  library(stringr)
  library(clusterProfiler)
  library(org.Mm.eg.db)
})

cortex <- load_cortex()
if (!"predicted_celltype" %in% colnames(cortex[[]])) {
  stop("predicted_celltype is missing. Run R/03_label_transfer.R.")
}
DefaultAssay(cortex) <- "Spatial"
Idents(cortex) <- "predicted_celltype"
out <- out_dir("celltypes")
blacklist <- c("Drd2", "Adora2a")

# Initial whole-isocortex screen. The reported isocortex test is the pseudobulk.
screen <- FindMarkers(
  cortex, ident.1 = "SMS", ident.2 = "CTR", group.by = "condition",
  logfc.threshold = 0.1
)
screen$gene <- rownames(screen)
screen <- screen[!screen$gene %in% blacklist, ]
screen <- screen[order(screen$avg_log2FC, decreasing = TRUE), ]
write.csv(screen, file.path(out, "whole_isocortex_wilcox.csv"), row.names = FALSE)

tables <- list()
for (ctype in unique(cortex$predicted_celltype)) {
  sub <- subset(cortex, idents = ctype)
  if (length(unique(sub$condition)) < 2) next
  markers <- tryCatch(
    FindMarkers(
      sub, ident.1 = "SMS", ident.2 = "CTR", group.by = "condition",
      logfc.threshold = 0.1, min.pct = 0.1
    ),
    error = function(e) NULL
  )
  if (is.null(markers) || nrow(markers) == 0) next
  markers$gene <- rownames(markers)
  markers$cell_type <- ctype
  markers <- markers[!markers$gene %in% blacklist, ]
  tables[[ctype]] <- markers
  write.csv(markers, file.path(out, paste0("markers_", gsub(" ", "_", ctype), ".csv")), row.names = FALSE)

  # |log2FC| > 0.25 and Bonferroni p < 0.05. The test itself used a threshold of 0.1.
  markers$call <- "NS"
  markers$call[markers$avg_log2FC > 0.25 & markers$p_val_adj < 0.05] <- "UP"
  markers$call[markers$avg_log2FC < -0.25 & markers$p_val_adj < 0.05] <- "DOWN"
  p <- ggplot(markers, aes(avg_log2FC, -log10(p_val_adj), color = call)) +
    geom_point(alpha = 0.6, size = 1.4) +
    scale_color_manual(values = c(DOWN = "blue", NS = "grey80", UP = "red")) +
    geom_vline(xintercept = c(-0.25, 0.25), linetype = "dashed") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
    theme_minimal() +
    labs(
      title = paste(ctype, "SMS versus CTR"),
      x = "log2 fold change",
      y = "-log10 Bonferroni p"
    ) +
    theme(legend.position = "none")
  ggsave(
    file.path(out, paste0("volcano_", gsub(" ", "_", ctype), ".png")),
    p, width = 7, height = 6, dpi = 140
  )

  for (direction in c("up", "down")) {
    if (direction == "up") {
      genes <- markers$gene[markers$avg_log2FC > 0 & markers$p_val_adj < 0.05]
    } else {
      genes <- markers$gene[markers$avg_log2FC < 0 & markers$p_val_adj < 0.05]
    }
    if (length(genes) <= 5) next
    ego <- enrichGO(genes, OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "BP",
                    pAdjustMethod = "BH", pvalueCutoff = 0.05)
    if (is.null(ego) || nrow(ego) == 0) next
    n_show <- min(15, nrow(ego))
    p <- dotplot(ego, showCategory = n_show) +
      ggtitle(paste(ctype, direction)) +
      scale_y_discrete(labels = function(x) stringr::str_wrap(x, width = 45))
    ggsave(
      file.path(out, paste0("go_", direction, "_", gsub(" ", "_", ctype), ".png")),
      p, width = 7, height = 4 + 0.25 * n_show, dpi = 140
    )
  }
}

all_markers <- bind_rows(tables)
write.csv(all_markers, file.path(out, "markers_all_celltypes.csv"), row.names = FALSE)

# Bonferroni p < 0.05 in four or more classes.
sig <- all_markers[all_markers$p_val_adj < 0.05, c("gene", "cell_type", "avg_log2FC")]
n_classes <- tapply(sig$cell_type, sig$gene, function(x) length(unique(x)))
shared_genes <- names(n_classes[n_classes >= 4])
if (length(shared_genes)) {
  shared <- sig[sig$gene %in% shared_genes, ]
  write.csv(shared, file.path(out, "genes_significant_in_four_or_more_classes.csv"), row.names = FALSE)
  p <- ggplot(shared, aes(cell_type, gene, fill = avg_log2FC)) +
    geom_tile() +
    scale_fill_gradient2(low = "#2C7BB6", mid = "white", high = "#D7191C") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(x = NULL, y = NULL, fill = "log2 FC", title = "Genes called in four or more classes")
  ggsave(file.path(out, "genes_shared_across_classes.png"), p, width = 8, height = 0.28 * length(shared_genes) + 2, dpi = 140)
}

message("Cell-type tables: ", nrow(all_markers), " rows")


## 6. Rai1 by class

`R/05_rai1_dotplot.R`


In [ ]:
# Rai1 by predicted class. Groups with fewer than 20 detected bins are drawn at the midpoint.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(ggplot2)
})

cortex <- load_cortex()
if (!"predicted_celltype" %in% colnames(cortex[[]])) {
  stop("predicted_celltype is missing. Run R/03_label_transfer.R.")
}

dat <- FetchData(cortex, vars = c("Rai1", "predicted_celltype", "condition"), layer = "data")
stats <- dat %>%
  group_by(predicted_celltype, condition) %>%
  summarise(
    bins = n(),
    bins_detected = sum(Rai1 > 0),
    percent_detected = 100 * sum(Rai1 > 0) / n(),
    mean_count = mean(expm1(Rai1)),
    .groups = "drop"
  )
stats$scaled <- as.numeric(scale(log1p(stats$mean_count)))
stats$color_value <- stats$scaled
# Fewer than 20 detected bins is drawn as the midpoint, not as a z-score.
stats$color_value[stats$bins_detected < 20] <- 0

out <- out_dir("celltypes")
write.csv(stats, file.path(out, "rai1_by_class.csv"), row.names = FALSE)

p <- ggplot(stats, aes(condition, predicted_celltype)) +
  geom_point(aes(size = percent_detected, color = color_value)) +
  scale_size_continuous(range = c(3, 9), limits = c(0, 100), name = "% bins detected") +
  scale_color_gradient2(
    low = "blue", mid = "grey95", high = "red", midpoint = 0,
    name = "Relative expression"
  ) +
  theme_minimal() +
  theme(axis.text.y = element_text(face = "italic")) +
  labs(x = NULL, y = NULL, title = "Rai1")
ggsave(file.path(out, "rai1_dotplot.png"), p, width = 6, height = 7, dpi = 140)


## 7. PV, SST, and VIP

`R/06_interneuron_subtypes.R`


In [ ]:
# Gad2-detected bins, clustered at resolution 0.4 on 15 PCs.
# A cluster is PV, SST, or VIP when that marker has the highest mean and the mean exceeds 0.1.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(ggrepel)
  library(dplyr)
  library(clusterProfiler)
  library(org.Mm.eg.db)
})

cortex <- load_cortex()
DefaultAssay(cortex) <- "Spatial"
out <- out_dir("subtypes")

inhib <- subset(cortex, subset = Gad2 > 0)
inhib <- NormalizeData(inhib, verbose = FALSE)
inhib <- FindVariableFeatures(inhib, verbose = FALSE)
inhib <- ScaleData(inhib, verbose = FALSE)
inhib <- RunPCA(inhib, verbose = FALSE)
set.seed(1)
inhib <- RunUMAP(inhib, dims = 1:15, verbose = FALSE)
inhib <- FindNeighbors(inhib, dims = 1:15, verbose = FALSE)
inhib <- FindClusters(inhib, resolution = 0.4, verbose = FALSE)

avg <- AverageExpression(inhib, features = c("Pvalb", "Sst", "Vip"), assays = "Spatial")$Spatial
if (all(grepl("^g[0-9]+$", colnames(avg)))) {
  colnames(avg) <- sub("^g", "", colnames(avg))
}

cluster_map <- data.frame(cluster = colnames(avg), subtype = "Other", stringsAsFactors = FALSE)
for (i in seq_len(ncol(avg))) {
  top <- rownames(avg)[which.max(avg[, i])]
  if (max(avg[, i]) <= 0.1) next
  cluster_map$subtype[i] <- switch(top, Pvalb = "PV", Sst = "SST", Vip = "VIP", "Other")
}
write.csv(cluster_map, file.path(out, "cluster_to_subtype.csv"), row.names = FALSE)

inhib$subtype <- cluster_map$subtype[match(as.character(Idents(inhib)), cluster_map$cluster)]
if (all(is.na(inhib$subtype))) stop("Cluster ids did not match the marker table.")
Idents(inhib) <- "subtype"
inhib <- subset(inhib, idents = c("PV", "SST", "VIP"))
print(table(inhib$subtype, inhib$condition))

p <- DimPlot(inhib, reduction = "umap", group.by = "subtype", label = TRUE) +
  ggtitle("Inhibitory bins")
ggsave(file.path(out, "umap_subtypes.png"), p, width = 6, height = 5, dpi = 140)
p <- DimPlot(inhib, reduction = "umap", group.by = "subtype", split.by = "condition") +
  ggtitle("Inhibitory bins")
ggsave(file.path(out, "umap_subtypes_by_condition.png"), p, width = 10, height = 5, dpi = 140)

# FindMarkers Bonferroni uses every gene. Subtype GO and volcanoes use the unadjusted p.
tables <- list()
for (st in c("PV", "SST", "VIP")) {
  sub <- subset(inhib, idents = st)
  if (length(unique(sub$condition)) < 2) next
  markers <- FindMarkers(
    sub, ident.1 = "SMS", ident.2 = "CTR", group.by = "condition",
    logfc.threshold = 0.05, min.pct = 0.05, verbose = FALSE
  )
  if (nrow(markers) == 0) next
  markers$gene <- rownames(markers)
  markers$cell_type <- st
  markers$direction <- ifelse(markers$avg_log2FC > 0, "UP", "DOWN")
  tables[[st]] <- markers
  write.csv(markers, file.path(out, paste0("markers_", st, ".csv")), row.names = FALSE)

  for (direction in c("UP", "DOWN")) {
    genes <- markers$gene[markers$direction == direction & markers$p_val < 0.05]
    if (length(genes) < 5) next
    ego <- enrichGO(genes, OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "BP",
                    pAdjustMethod = "BH", pvalueCutoff = 0.05)
    if (is.null(ego) || nrow(ego) == 0) next
    gp <- dotplot(ego, showCategory = 10) + ggtitle(paste(st, direction))
    ggsave(file.path(out, paste0("go_", tolower(direction), "_", st, ".png")), gp, width = 6, height = 6, dpi = 140)
  }
}
all_sub <- bind_rows(tables)
write.csv(all_sub, file.path(out, "markers_all_subtypes.csv"), row.names = FALSE)

# Rai1 is labeled on every subtype volcano, with the immediate-early genes.
down_core <- c("Fos", "Junb", "Fosl2", "Egr1", "Egr2", "Nr4a1", "Nr4a2", "Nr4a3", "Arc", "Dusp5", "Rai1")
up_list <- list(
  VIP = c("Scg5", "Rbfox1", "Plch2", "Btbd9", "Camk2n1"),
  SST = c("Camk2n1", "Glo1", "mt-Co2", "Nrn1", "Scn1b"),
  PV = c("Glo1", "mt-Co2", "Scn1b", "Camk2n1", "Nrn1")
)
for (st in names(tables)) {
  df <- tables[[st]]
  df$color <- "NS"
  df$color[df$avg_log2FC > 0.05 & df$p_val < 0.05] <- "UP"
  df$color[df$avg_log2FC < -0.05 & df$p_val < 0.05] <- "DOWN"
  label_genes <- intersect(df$gene, c(down_core, up_list[[st]]))
  df$label <- ifelse(df$gene %in% label_genes, df$gene, NA)
  p <- ggplot(df, aes(avg_log2FC, -log10(p_val), color = color, label = label)) +
    geom_point(alpha = 0.6, size = 1.4) +
    scale_color_manual(values = c(DOWN = "blue", NS = "grey80", UP = "red")) +
    geom_vline(xintercept = c(-0.05, 0.05), linetype = "dashed") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed") +
    geom_text_repel(max.overlaps = 50, force = 2, size = 3, min.segment.length = 0) +
    theme_minimal() +
    labs(title = paste(st, "SMS versus CTR"), x = "log2 fold change", y = "-log10 p") +
    theme(legend.position = "none")
  ggsave(file.path(out, paste0("volcano_", st, ".png")), p, width = 7, height = 6, dpi = 140)
}

cortex$subtype <- NA_character_
cortex$subtype[match(colnames(inhib), colnames(cortex))] <- as.character(inhib$subtype)
save_cortex(cortex)
saveRDS(inhib, file.path(results_dir, "inhib.rds"))
message("Subtype bins: ", ncol(inhib))


## 8. CellChat

`R/07_cellchat.R`


In [ ]:
# Ligand-receptor inference for excitatory, PV, SST, and VIP bins.
# CTR and SMS are fit separately. min.cells = 3 because several subtypes are small.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(CellChat)
})

cortex <- load_cortex()
if (!"subtype" %in% colnames(cortex[[]])) stop("subtype is missing. Run R/06_interneuron_subtypes.R.")

cortex$fine_label <- as.character(cortex$predicted_celltype)
has_subtype <- !is.na(cortex$subtype)
cortex$fine_label[has_subtype] <- cortex$subtype[has_subtype]

keep <- cortex$fine_label %in% c("Excitatory", "PV", "SST", "VIP")
mini <- subset(cortex, cells = colnames(cortex)[keep])
mini$fine_label <- factor(mini$fine_label, levels = c("Excitatory", "PV", "SST", "VIP"))
Idents(mini) <- "fine_label"

fit_cellchat <- function(obj) {
  obj$fine_label <- droplevels(factor(
    obj$fine_label, levels = c("Excitatory", "PV", "SST", "VIP")
  ))
  obj <- JoinLayers(obj)
  expr <- GetAssayData(obj, assay = "Spatial", layer = "data")
  cc <- createCellChat(object = expr, meta = obj@meta.data, group.by = "fine_label")
  cc@DB <- CellChatDB.mouse
  cc <- subsetData(cc)
  cc <- identifyOverExpressedGenes(cc)
  cc <- identifyOverExpressedInteractions(cc)
  cc <- computeCommunProb(cc, type = "triMean")
  cc <- filterCommunication(cc, min.cells = 3)
  cc <- computeCommunProbPathway(cc)
  cc <- aggregateNet(cc)
  if (sum(cc@net$count) == 0) {
    message("No interactions at min.cells = 3. Refitting without that filter.")
    cc <- computeCommunProb(cc, type = "triMean")
    cc <- computeCommunProbPathway(cc)
    cc <- aggregateNet(cc)
  }
  cc
}

ctr <- fit_cellchat(subset(mini, subset = condition == "CTR"))
sms <- fit_cellchat(subset(mini, subset = condition == "SMS"))
saveRDS(list(CTR = ctr, SMS = sms), file.path(results_dir, "cellchat.rds"))

out <- out_dir("cellchat")
colors <- scPalette(4)
names(colors) <- c("Excitatory", "PV", "SST", "VIP")
for (nm in c("CTR", "SMS")) {
  cc <- if (nm == "CTR") ctr else sms
  message(nm, " interactions: ", sum(cc@net$count))
  if (sum(cc@net$count) == 0) next
  png(file.path(out, paste0("circle_", nm, ".png")), width = 2000, height = 2000, res = 300)
  netVisual_circle(
    cc@net$count,
    vertex.weight = as.numeric(table(cc@idents)),
    weight.scale = TRUE,
    label.edge = FALSE,
    color.use = colors[levels(cc@idents)],
    title.name = nm
  )
  dev.off()
}


## 9. Slingshot

`R/08_slingshot.R`


In [ ]:
# Shared pseudotime for CTR and SMS. Inhibitory bins use the subtype labels.
# Excitatory bins are reclustered at resolution 0.3 to give Slingshot anchor groups.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(SingleCellExperiment)
  library(slingshot)
  library(ggplot2)
  library(viridis)
})

draw_trajectory <- function(df, curves, title, path) {
  p <- ggplot() +
    geom_point(data = df, aes(UMAP_1, UMAP_2, color = Pseudotime), size = 0.8, alpha = 0.7) +
    scale_color_viridis_c(option = "magma", na.value = "grey90") +
    geom_path(data = curves, aes(UMAP_1, UMAP_2, group = Lineage),
              color = "black", linewidth = 0.8,
              arrow = arrow(length = unit(0.25, "cm"))) +
    facet_wrap(~ Condition) +
    theme_minimal() +
    labs(title = title, color = "Pseudotime")
  ggsave(path, p, width = 10, height = 5, dpi = 140)

  dens <- ggplot(df, aes(Pseudotime, fill = Condition)) +
    geom_density(alpha = 0.5) +
    scale_fill_manual(values = c(CTR = "grey60", SMS = "red")) +
    theme_minimal() +
    labs(title = title, x = "Pseudotime", y = "Density")
  ggsave(sub("trajectory", "density", path), dens, width = 7, height = 4, dpi = 140)
}

curve_frame <- function(sce) {
  curves <- slingCurves(sce)
  bind <- lapply(seq_along(curves), function(i) {
    xy <- curves[[i]]$s[curves[[i]]$ord, , drop = FALSE]
    data.frame(UMAP_1 = xy[, 1], UMAP_2 = xy[, 2], Lineage = paste0("Lineage", i))
  })
  do.call(rbind, bind)
}

out <- out_dir("trajectory")

inhib_path <- file.path(results_dir, "inhib.rds")
if (!file.exists(inhib_path)) stop("inhib.rds is missing. Run R/06_interneuron_subtypes.R.")
inhib <- readRDS(inhib_path)
set.seed(1)
sce <- slingshot(as.SingleCellExperiment(inhib), clusterLabels = "subtype", reducedDim = "UMAP")
pt <- slingPseudotime(sce)
umap <- reducedDims(sce)$UMAP
df <- data.frame(
  UMAP_1 = umap[, 1], UMAP_2 = umap[, 2],
  Pseudotime = pt[, 1],
  Condition = inhib$condition,
  Subtype = inhib$subtype
)
draw_trajectory(df, curve_frame(sce), "Inhibitory subtypes", file.path(out, "inhibitory_trajectory.png"))

cortex <- load_cortex()
Idents(cortex) <- "predicted_celltype"
excit <- subset(cortex, idents = "Excitatory")
DefaultAssay(excit) <- "Spatial"
excit <- FindVariableFeatures(excit, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
excit <- ScaleData(excit, verbose = FALSE)
excit <- RunPCA(excit, verbose = FALSE)
set.seed(1)
excit <- RunUMAP(excit, dims = 1:30, verbose = FALSE)
excit <- FindNeighbors(excit, dims = 1:30, verbose = FALSE)
excit <- FindClusters(excit, resolution = 0.3, verbose = FALSE)

set.seed(1)
sce <- slingshot(as.SingleCellExperiment(excit), clusterLabels = "seurat_clusters", reducedDim = "UMAP")
pt <- slingPseudotime(sce)
umap <- reducedDims(sce)$UMAP
df <- data.frame(
  UMAP_1 = umap[, 1], UMAP_2 = umap[, 2],
  Pseudotime = pt[, 1],
  Condition = excit$condition,
  Cluster = excit$seurat_clusters
)
draw_trajectory(df, curve_frame(sce), "Excitatory bins", file.path(out, "excitatory_trajectory.png"))
message("Trajectory plots are in ", out)


## 10. Pia-inward depth

`R/09_cortical_depth.R`


In [ ]:
# Depth is distance to the pial contour, divided by the piece-wise 99.5th percentile.
# Faces of the crop rectangle are not treated as pia.
source("R/00_config.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(FNN)
})

dilate4 <- function(z) {
  nr <- nrow(z)
  nc <- ncol(z)
  z |
    rbind(z[-1, , drop = FALSE], FALSE) |
    rbind(FALSE, z[-nr, , drop = FALSE]) |
    cbind(z[, -1, drop = FALSE], FALSE) |
    cbind(FALSE, z[, -nc, drop = FALSE])
}

flood_exterior <- function(tissue) {
  nr <- nrow(tissue)
  nc <- ncol(tissue)
  ext <- matrix(FALSE, nr, nc)
  ext[1, ] <- !tissue[1, ]
  ext[nr, ] <- !tissue[nr, ]
  ext[, 1] <- !tissue[, 1]
  ext[, nc] <- !tissue[, nc]
  for (i in seq_len(4000)) {
    nxt <- dilate4(ext) & !tissue
    if (identical(nxt, ext)) break
    ext <- nxt
  }
  ext
}

cortex <- load_cortex()
cortex$depth_from_pia <- NA_real_
cortex$layer_band <- NA_character_
cortex$layer_class <- NA_character_
pia_list <- list()
bin_list <- list()

for (rep in sort(unique(cortex$spatial_replicate))) {
  cells <- colnames(cortex)[cortex$spatial_replicate == rep]
  xy <- data.frame(x = cortex$pxl_x[cells], y = cortex$pxl_y[cells])
  step <- median(FNN::get.knnx(as.matrix(xy), as.matrix(xy), k = 2)$nn.dist[, 2])
  if (!is.finite(step) || step <= 0) step <- 56.25
  x0 <- min(xy$x)
  y0 <- min(xy$y)
  jj <- as.integer(round((xy$x - x0) / step)) + 3L
  ii <- as.integer(round((xy$y - y0) / step)) + 3L
  tissue <- matrix(FALSE, max(ii) + 2L, max(jj) + 2L)
  tissue[cbind(ii, jj)] <- TRUE
  outer_b <- tissue & dilate4(flood_exterior(tissue))
  ob <- which(outer_b, arr.ind = TRUE)
  bound <- data.frame(
    x = x0 + (ob[, 2] - 3) * step,
    y = y0 + (ob[, 1] - 3) * step
  )
  rx <- range(xy$x)
  ry <- range(xy$y)
  pad <- 2.2 * step
  on_crop <- bound$x <= rx[1] + pad | bound$x >= rx[2] - pad |
    bound$y <= ry[1] + pad | bound$y >= ry[2] - pad
  pia <- bound[!on_crop, , drop = FALSE]
  if (nrow(pia) < 2) stop("Pial contour is empty for ", rep)
  dist_pia <- FNN::get.knnx(as.matrix(pia), as.matrix(xy), k = 1)$nn.dist[, 1]
  depth <- pmin(pmax(dist_pia / (as.numeric(quantile(dist_pia, 0.995)) + 1e-6), 0), 1)
  band <- ifelse(depth <= cut_L1, "L1",
          ifelse(depth <= cut_L23, "L2/3",
          ifelse(depth <= cut_L5, "L5", "L6")))
  cls <- ifelse(depth <= upper_max, "upper", ifelse(depth >= deep_min, "deep", NA_character_))
  cortex$depth_from_pia[cells] <- depth
  cortex$layer_band[cells] <- band
  cortex$layer_class[cells] <- cls
  pia_list[[rep]] <- data.frame(pia, replicate = rep)
  bin_list[[rep]] <- data.frame(x = xy$x, y = xy$y, depth = depth, band = band, replicate = rep)
  message(rep, "  L1 ", sum(band == "L1"), "  L2/3 ", sum(band == "L2/3"),
          "  L5 ", sum(band == "L5"), "  L6 ", sum(band == "L6"))
}

# These L1 counts are from the executed run. A mismatch means the contour moved.
expected_l1 <- c(CTR_Rep1 = 5397L, CTR_Rep2 = 4540L, SMS_Rep1 = 3404L, SMS_Rep2 = 4308L)
got_l1 <- sapply(names(expected_l1), function(rep) {
  sum(cortex$layer_band == "L1" & cortex$spatial_replicate == rep, na.rm = TRUE)
})
if (!identical(as.integer(got_l1), as.integer(expected_l1))) {
  stop("L1 counts changed: ", paste(got_l1, collapse = ", "))
}

out <- out_dir("depth")
pia_df <- do.call(rbind, pia_list)
bin_df <- do.call(rbind, bin_list)
write.csv(
  as.data.frame(table(cortex$layer_band, cortex$spatial_replicate)),
  file.path(out, "layer_band_counts.csv"),
  row.names = FALSE
)

theme_sp <- theme_minimal() + theme(legend.position = "bottom", strip.text = element_text(face = "bold"))
p_pia <- ggplot() +
  geom_point(data = bin_df, aes(x, y), color = "grey82", size = 0.12) +
  geom_point(data = pia_df, aes(x, y), color = "#00BFC4", size = 0.4) +
  scale_y_reverse() +
  facet_wrap(~replicate, scales = "free") +
  theme_sp +
  labs(title = "Pial contour", x = NULL, y = NULL)
ggsave(file.path(out, "pial_contour.png"), p_pia, width = 12, height = 9, dpi = 140)

p_band <- ggplot(bin_df, aes(x, y, color = band)) +
  geom_point(size = 0.15) +
  scale_color_manual(values = c("L1" = "#F4A261", "L2/3" = "#2C7BB6", "L5" = "#D7191C", "L6" = "#6A3D9A")) +
  scale_y_reverse() +
  facet_wrap(~replicate, scales = "free") +
  theme_sp +
  labs(title = "Display bands", x = NULL, y = NULL, color = NULL)
ggsave(file.path(out, "layer_bands.png"), p_band, width = 12, height = 9, dpi = 140)

markers <- c("Reln", "Lamp5", "Cux2", "Rorb", "Ndnf", "Sncg", "Pvalb", "Nefm", "Bcl11b", "Fezf2", "Foxp2", "Tbr1")
markers <- intersect(markers, rownames(cortex))
expr <- as.matrix(GetAssayData(cortex, layer = "data")[markers, , drop = FALSE])
long <- do.call(rbind, lapply(markers, function(g) {
  data.frame(
    replicate = cortex$spatial_replicate,
    depth = cortex$depth_from_pia,
    gene = g,
    expr = expr[g, ],
    stringsAsFactors = FALSE
  )
}))
p_san <- ggplot(long, aes(depth, expr, color = gene)) +
  geom_smooth(method = "loess", span = 0.35, se = FALSE, linewidth = 0.7, formula = y ~ x) +
  facet_wrap(~replicate, ncol = 2) +
  theme_minimal() +
  labs(x = "Depth from pia", y = "Log-normalized expression", title = "Laminar markers")
ggsave(file.path(out, "markers_vs_depth.png"), p_san, width = 12, height = 8, dpi = 140)

save_cortex(cortex)
message("Depth written. L1 counts match the executed run.")


## 11. DiSTect model

`R/distect_model.R`


In [ ]:
# DiSTect autologistic model. Y = 1 for SMS and 0 for CTR.
# A positive coefficient is higher expression in SMS.
# The last two columns of x are coordinates and are used only for the neighbor test.
# The radius is 1 coordinate unit. On full-resolution pixels, neighboring 16 um bins
# are about 56 px apart, so that term stays off. Array row and column turn some of it on.

stan_program <- "
data {
  int<lower=0> N; int<lower=0> P; int<lower=0> S;
  matrix[N,P+2] x; int y[N]; vector[S] u_mu; int label[N];
}
parameters {
  vector[P] beta; real<lower=0,upper=8> eta; vector<lower=0>[P] beta_gamma;
  real<lower=0,upper=1> w; vector<lower=-1,upper=1>[S] U;
  real<lower=0,upper=1> sigma_square; real<lower=0,upper=1> rho;
}
model{
  vector[N] mu; matrix[N,N] prob_neigh; matrix[S,S] covar_matrix;
  rho ~ uniform(0,1); sigma_square ~ inv_gamma(5,50);
  for (i in 1:S){ for (j in 1:S){
    if (i == j){ covar_matrix[i,j]=sigma_square; } else { covar_matrix[i,j]=sigma_square*rho; }
  }}
  U ~ multi_normal(u_mu, covar_matrix);
  for(i in 1:P){
    w ~ uniform(0,1); beta_gamma[i] ~ inv_gamma(5,50);
    target += log_sum_exp(log(1-w)+normal_lpdf(beta[i]|0,0.000001*beta_gamma[i]), log(w)+normal_lpdf(beta[i]|0,beta_gamma[i]));
  }
  for(i in 1:N) {
    for(j in 1:N){
      eta ~ uniform(0,8);
      if (label[i]==label[j] && j != i && sqrt(square(x[i,P+1]-x[j,P+1])+square(x[i,P+2]-x[j,P+2]))<=1){
        prob_neigh[i,j]=eta*y[j];
      }else{ prob_neigh[i,j]=0; }
    }
    mu[i] = dot_product(x[i,1:P],beta)+sum(prob_neigh[i,])+U[label[i]];
  }
  y ~ bernoulli_logit(mu);
}
"

plot_sms_ctr <- function(dat, title, fname) {
  dat$gene <- factor(dat$gene, levels = dat$gene)
  show_n <- nrow(dat)
  p <- ggplot2::ggplot(dat, ggplot2::aes(gene, posterior_mean, color = direction)) +
    ggplot2::geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
    ggplot2::geom_pointrange(ggplot2::aes(ymin = cri_2.5, ymax = cri_97.5), linewidth = 0.3) +
    ggplot2::scale_color_manual(values = c(higher_in_SMS = "#D7191C", higher_in_CTR = "#2C7BB6")) +
    ggplot2::theme_bw() +
    ggplot2::theme(
      axis.text.x = ggplot2::element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6),
      legend.position = "top"
    ) +
    ggplot2::labs(
      title = title,
      subtitle = "Red / positive = higher in SMS. Blue / negative = higher in CTR. Bars = 95% CrI.",
      x = NULL, y = "Posterior mean", color = NULL
    )
  ggplot2::ggsave(fname, p, width = max(10, 0.28 * show_n + 4), height = 5.8, dpi = 140)
  invisible(p)
}

run_sms_ctr <- function(obj, cells, tag, max_per_rep, out, stan_mod) {
  set.seed(42)
  md <- obj[[]][cells, , drop = FALSE]
  sampled <- unlist(lapply(unique(as.character(md$spatial_replicate)), function(rep) {
    ids <- cells[as.character(md$spatial_replicate) == rep]
    sample(ids, min(length(ids), max_per_rep))
  }), use.names = FALSE)

  sub <- subset(obj, cells = sampled)
  sub <- Seurat::JoinLayers(sub)
  sub <- Seurat::FindVariableFeatures(sub, selection.method = "vst", nfeatures = 200, verbose = FALSE)
  hvg <- Seurat::VariableFeatures(sub)
  expr <- t(as.matrix(Seurat::GetAssayData(sub, layer = "data")[hvg, sampled, drop = FALSE]))
  matrix_x <- as.matrix(cbind(
    expr,
    x = obj$coord_x[sampled],
    y = obj$coord_y[sampled]
  ))
  y_vector <- as.integer(obj$condition[sampled] == "SMS")
  label_list <- as.integer(factor(obj$spatial_replicate[sampled]))
  ok <- complete.cases(matrix_x)
  matrix_x <- matrix_x[ok, , drop = FALSE]
  y_vector <- y_vector[ok]
  label_list <- label_list[ok]

  fit <- rstan::vb(
    stan_mod,
    data = list(
      N = nrow(matrix_x), P = ncol(matrix_x) - 2, x = matrix_x, y = y_vector,
      label = label_list, S = length(unique(label_list)),
      u_mu = rep(0, length(unique(label_list)))
    ),
    algorithm = "meanfield", seed = 128, tol_rel_obj = 1e-04, iter = 10000
  )
  beta <- rstan::summary(fit)$summary
  beta <- beta[grep("^beta\\[", rownames(beta)), , drop = FALSE]
  n_genes <- ncol(matrix_x) - 2
  coef_df <- data.frame(
    gene = colnames(matrix_x)[seq_len(n_genes)],
    posterior_mean = beta[seq_len(n_genes), "mean"],
    cri_2.5 = beta[seq_len(n_genes), "2.5%"],
    cri_97.5 = beta[seq_len(n_genes), "97.5%"],
    stringsAsFactors = FALSE
  )
  coef_df$excludes_zero <- (coef_df$cri_97.5 < 0) | (coef_df$cri_2.5 > 0)
  coef_df$direction <- ifelse(coef_df$posterior_mean > 0, "higher_in_SMS", "higher_in_CTR")
  coef_df <- coef_df[order(-abs(coef_df$posterior_mean)), ]
  utils::write.csv(coef_df, file.path(out, paste0(tag, "_coefficients.csv")), row.names = FALSE)
  selected <- coef_df[coef_df$excludes_zero, ]
  utils::write.csv(selected, file.path(out, paste0(tag, "_selected.csv")), row.names = FALSE)
  message(tag, ": ", nrow(selected), " / ", nrow(coef_df), " genes called")
  if (nrow(selected)) {
    plot_sms_ctr(
      utils::head(selected, 40),
      paste0("SMS vs CTR, ", tag),
      file.path(out, paste0(tag, "_cri.png"))
    )
  }
  invisible(coef_df)
}

fit_three <- function(obj, out) {
  if (!exists("distect_stan_mod", inherits = TRUE)) {
    distect_stan_mod <<- rstan::stan_model(model_code = stan_program)
  }
  all_cells <- colnames(obj)
  upper <- colnames(obj)[!is.na(obj$layer_class) & obj$layer_class == "upper"]
  deep <- colnames(obj)[!is.na(obj$layer_class) & obj$layer_class == "deep"]
  run_sms_ctr(obj, all_cells, "all_cortex", 500, out, distect_stan_mod)
  run_sms_ctr(obj, upper, "upper_L1_L23", 400, out, distect_stan_mod)
  run_sms_ctr(obj, deep, "deep_L5_L6", 400, out, distect_stan_mod)
}


## 12. DiSTect, pixel coordinates

`R/10_distect_pixel.R`


In [ ]:
# Pixel-coordinate DiSTect. This is the fit behind the current credible-interval figure.
# Each model is mean-field ADVI and takes roughly 15 minutes.
source("R/00_config.R")
source("R/distect_model.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(rstan)
  library(ggplot2)
})
rstan_options(auto_write = TRUE)

cortex <- load_cortex()
if (!"depth_from_pia" %in% colnames(cortex[[]])) {
  stop("Depth is missing. Run R/09_cortical_depth.R.")
}
cortex$coord_x <- cortex$pxl_x
cortex$coord_y <- cortex$pxl_y
fit_three(cortex, out_dir("distect"))


## 13. DiSTect, array row and column

`R/11_distect_array.R`


In [ ]:
# Same bins and the same seeds as R/10_distect_pixel.R.
# Neighbor distance is Space Ranger array row and column instead of image pixels.
source("R/00_config.R")
source("R/distect_model.R")

suppressPackageStartupMessages({
  library(Seurat)
  library(rstan)
  library(ggplot2)
})
rstan_options(auto_write = TRUE)

cortex <- load_cortex()
if (!"depth_from_pia" %in% colnames(cortex[[]])) {
  stop("Depth is missing. Run R/09_cortical_depth.R.")
}
if (anyNA(cortex$array_row) || anyNA(cortex$array_col)) {
  stop("Array coordinates are missing.")
}
cortex$coord_x <- cortex$array_row
cortex$coord_y <- cortex$array_col
fit_three(cortex, out_dir("distect_array"))
